In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
def stumpf_S(z):
    """
    Stumpf function S(z).

    TODO: figure out what tolerance for |z|>tol I actually should use
    """
    if z > 1e-12:
        sqrt_z = np.sqrt(z)
        return (sqrt_z - np.sin(sqrt_z)) / (sqrt_z**3)
    elif z < -1e-12:
        sqrt_neg_z = np.sqrt(-z)
        return (np.sinh(sqrt_neg_z) - sqrt_neg_z) / (sqrt_neg_z**3)
    else:
        return 1/6

def stumpf_C(z):
    """
    Stumpf function C(z).

    TODO: figure out what tolerance for |z|>tol I actually should use
    """
    if z > 1e-12:
        sqrt_z = np.sqrt(z)
        return (1 - np.cos(sqrt_z)) / z
    elif z < -1e-12:
        sqrt_neg_z = np.sqrt(-z)
        return (np.cosh(sqrt_neg_z) - 1) / (-z)
    else:
        return 1/2
    
def stumpf_dSdz(z):
    """
    Stumpf function S(z) derivative with respect to z.
    """
    if math.abs(z) < 1e-2:
        # use power series expansion for small z
        return -1/math.factorial(5) + 2*z/math.factorial(7) - 3*z**2/math.factorial(9) + 4*z**3/math.factorial(11)
    
    S = stumpf_S(z)
    C = stumpf_C(z)
    
    return (C - 3*S) / (2*z)

def stumpf_dCdz(z):
    """
    Stumpf function C(z) derivative with respect to z.
    """
    if np.abs(z) < 1e-2:
        # use power series expansion for small z
        return -1/math.factorial(4) + 2*z/math.factorial(6) - 3*z**2/math.factorial(8) + 4*z**3/math.factorial(10)

    S = stumpf_S(z)
    C = stumpf_C(z)

    return (1 - z*S - 2*C) / (2*z)

In [ ]:
def gauss_uv(r1_vec, r2_vec, dt=0, mu=0):
    # standarize inputs
    r1_vec = np.array(r1_vec, dtype=float)
    r2_vec = np.array(r2_vec, dtype=float)

    r1 = np.linalg.norm(r1_vec)
    r2 = np.linalg.norm(r2_vec)
    

    # step 0: compute nu, angle between two vectors
    #   a dot b     = |a|*|b|*cos(nu)
    #   |a cross b| = |a|*|b|*sin(nu)
    #   tan(nu) =  |a cross b| / a dot b
    nu_short = math.atan2(
        np.linalg.norm(np.cross(r1_vec, r2_vec)), 
        r1_vec.dot(r2_vec)
    )
    nu_long = 2*np.pi - nu_short

    # short way
    (f, g, gdot) = gauss_uv_fg_solver(r1, r2, dt, mu, nu_short)
    v1_short = (r2_vec - f*r1_vec) / g
    v2_short = (gdot*r2_vec - r1_vec) / g

    # long way
    (f, g, gdot) = gauss_uv_fg_solver(r1, r2, dt, mu, nu_long)
    v1_long = (r2_vec - f*r1_vec) / g
    v2_long = (gdot*r2_vec - r1_vec) / g

    # TODO: get perigee values and eliminate ones that cross through earth... or maybe do this by consumer of this function

    return v1_short, v2_short


def gauss_uv_fg_solver(
    r1: float, 
    r2: float, 
    dt: float, 
    mu: float, 
    nu: float
) -> tuple[float, float, float]:
    # direction of motion
    DM = np.sign(np.pi - nu)

    # step 1: from r1 and r2 and "direction of motion", evaluate the constant (eq. 5-15, eq. 5-37 BMW)
    A = DM * math.sqrt(r1 * r2 * (1 + math.cos(nu)))

    # step 2: pick a trial value for z
    #         z = deltaE^2 for elliptical
    #         z = deltaF^2 for hyperolic
    #         good initial guess is z = 0
    z = 0
    dz = 0

    # newton raphson parameters
    dt_tol = 1e-4
    nr_max_iter = 100
    nr_counter = 0

    # bisection search parameters
    bs_max_iter = 100
    bs_counter = 0

    while(True):
        if nr_counter >= nr_max_iter:
            print("Warning: max newton-raphson iterations reached when solving for z")
            break
        if bs_counter >= bs_max_iter:
            print("Warning: max bisection-search iterations reached when solving for z")
            break
        
        bs_counter += 1

        z_trial = z + dz

        # step 3: evaluate S and C for selected z (eq. 4-37 and eq. 4-38 BMW)
        S = stumpf_S(z_trial)
        C = stumpf_C(z_trial)

        # step 4: determine aux variable y (eq. 5-17 BMW)
        y = r1 + r2 - A * (1 - z_trial*S) / math.sqrt(C)

        # step 4.5: check if y is negative (only can happen with short way trajectories)
        #           keep halving step size until y is no longer negative
        if y < 0:
            dz = 0.5*dz
            continue
        
        # z_trial is accepted
        z = z_trial

        nr_counter += 1
        bs_counter = 0

        # step 5: determine x (eq. 5-18 BMW)
        x = math.sqrt(y / C)

        # step 6: check trial value of z by computing dt_trial (eq. 5-20 BMW)
        #         then compare to true dt 
        #         then newton-raphson iterate

        dt_trial = (S*x**3 + A*math.sqrt(y)) / math.sqrt(mu)

        if np.abs(dt - dt_trial) < dt_tol:
            break

        # TODO:
        dz = 0

    # step 7: evaluate f, g, gdot (eq. 5-21, 5-22, 5-23 BMW)
    #         then compute v1 and v2 (eq. 5-24, 5-25 BMW)

    S = stumpf_S(z)
    C = stumpf_C(z)
    y = r1 + r2 - A * (1 - z*S) / math.sqrt(C)

    f = 1 - y/r1
    g = A*math.sqrt(y/mu)
    gdot = 1 - y/r2

    return (f, g, gdot)


# debugging
# r1 = [1, 0.1, 0]
# r2 = [0.1, 0, 0]
# gauss_uv(r1, r2)